<a href="https://colab.research.google.com/github/elhamod/IS834_Fall_2026/blob/main/Session%2005%20-%20Pandas%20III%20-%20Aggregation%20and%20Reshaping/05_Analytics_with_pandas_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IS834 — Session 5

Work through this notebook in Colab. The lecture cells are filled in and ready to run; the
**Warmups** and **Exercises** are yours to complete — write your code in the `# Your code here` cells and your
reasoning in the `*Provide your answer here*` cells.

Every dataset is read straight from the course repository, so there is nothing to upload or install.

*Adapted from Brock Tibert's IS834 Fall 2025 session 5. Edited for Fall 2026 by Mohannad Elhamod.*

# Learning Goals

Sessions 3 and 4 got one dataset loaded, described and cleaned. All of that was preparation for a
question you cannot answer a row at a time: *how do the groups compare?*

- Group summaries with `groupby`
- `.count()` versus `.size()`, and why they disagree
- Aggregating several columns, and named aggregates
- Cross-tabulations and pivot tables
- Reshaping: wide to long and back with `.melt()`
- Choosing between `groupby`, `crosstab` and `pivot_table`

A look ahead:

- **Quiz 2** at the start of next session (session 6), handwritten, same format as Quiz 1
- Plotting and exploratory data analysis (session 6)
- A full end-to-end practice lab (session 7) — load, clean, aggregate, visualize, communicate
- Combining datasets with joins and merges (session 8)

# Setup


In [ ]:
# pandas is the only package this notebook needs, and Colab already has it installed.
import pandas as pd

# Every data file this notebook uses is read straight from the course repository.
# There is nothing to upload: DATA is the folder those files live in.
DATA = "https://raw.githubusercontent.com/elhamod/IS834_Fall_2026/main/data/"


In [ ]:
# Load the diamonds dataset, used in the practice exercise later in the session.
diamonds = pd.read_csv(DATA + "diamonds.csv")

print(diamonds.shape)

**Tip.** `%whos` lists every variable currently sitting in memory — useful when you lose track of what a notebook has already defined.

In [ ]:
# List the variables currently in memory.
%whos

# Warmup Exercise!


**Warmup 1.** Read the `Orders` worksheet from the `sales.xlsx` workbook.

In [ ]:
# Your code here

**Warmup 2.** Return the information about the dataset. Are there any missing values?

In [ ]:
# Your code here

**Answer**

*Provide your answer here*

**Warmup 3.** Summarize the numeric variables in the dataset.

In [ ]:
# Your code here

**Answer**

*Provide your answer here*

> Session 3 answered "which region is struggling?" with four filtered DataFrames and a loop, and
> session 4 spent its time making a file trustworthy enough to ask the question of. Today we do the
> same summary in a single line — and then everything that a single line makes newly worth asking.

# Group Summaries


## GroupBy

`groupby` works in three steps: **split** the rows into groups by the values of a column, **apply** an aggregation (a count, a sum, a mean) to each group, and **combine** the results into one small table.

![](https://www.w3resource.com/w3r_images/pandas-groupby-split-apply-combine.svg)

The same idea with more than one grouping column — every combination of values becomes its own group:

![](https://miro.medium.com/v2/resize:fit:1400/0*m-puRNwcnUl5zZ4O)

**Tidy the column names first.** The warmup names have capitals and spaces, so `orders.Ship Mode` cannot work — a name with a space can only be reached with brackets and quotes. Renaming once at the top means every column can be written the same short way.

In [ ]:
# Lowercase the column names and replace spaces with underscores.
# Reading the worksheet again here means this cell works whether or not you finished Warmup 1.
orders = pd.read_excel(DATA + "sales.xlsx", sheet_name="Orders")
orders.columns = orders.columns.str.replace(" ", "_").str.lower()
orders.columns.tolist()

`value_counts()` is the simple version of a groupby: it groups the rows by one column and counts how many fall in each group.

In [ ]:
# Count the orders per ship mode.
x = orders.ship_mode.value_counts()
x

pandas gives both rows and columns an **index** — a set of labels. In the result above, the ship modes have become the row labels:

In [ ]:
# The row labels (the index) of x.
x.index

`groupby` is the general tool. But on its own it is "greedy": ask it to `.count()` and it counts every column it can.

In [ ]:
# Group by ship mode and count -- every column at once.
orders.groupby("ship_mode").count()

> That is 23 columns of counts when we wanted one. It is not wrong, just unfocused — and on a wide table it is slow. **Always tell `groupby` which column you want aggregated.**


In [ ]:
# Replicating value_counts: group by ship_mode, keep the ship_mode column, count rows, sort descending.
orders.groupby("ship_mode")['ship_mode'].count().sort_values(ascending=False)


### The aggregation you pick is a decision, not a formality

- `.count()` counts values that are **present**; `.size()` counts **rows**, missing or not.
- `.mean()`, `.median()`, `.min()`, `.max()`, `.sum()`, `.std()` all work the same way.
- pandas is built on top of numpy, so in examples online you will often see `np.mean` and friends passed in where a plain string like `"mean"` does the same job. Prefer the string — it is shorter and it keeps numpy out of your imports.

In [ ]:
# Rows per ship mode (.size) next to margins recorded per ship mode (.count).
grouped = orders.groupby("ship_mode")

pd.DataFrame({'rows (size)': grouped.size(),
              'margins present (count)': grouped['product_base_margin'].count()})

**Read the two columns against each other.** They disagree by exactly the 72 missing product margins we found in the warmup. `.size()` answers "how many orders shipped this way?"; `.count()` answers "how many of those have a margin recorded?". Reaching for the wrong one gives you a denominator that is quietly too small, which is how a "percentage of orders" turns into a "percentage of orders where the field happened to be filled in".


By default the group labels become the **index** of the result. `as_index=False` keeps them as an ordinary column instead, and returns a DataFrame.

In [ ]:
# Count orders per ship mode, keeping ship_mode as an ordinary column.
xdf = orders.groupby("ship_mode", as_index=False)['row_id'].count()
print(type(xdf))
xdf

> This is the form to default to. With the group labels as a normal column, the result behaves like any other table — you can filter it, merge it, plot it or write it to Excel without first remembering `.reset_index()`.


**Exercise 1.** Group by shipping mode **and** customer segment, and count the orders in each combination. *Hint:* a single grouping column is passed as a string; several are passed as a list of strings.

In [ ]:
# Your code here

**Answer**

*Provide your answer here*

In [ ]:
# Two grouping columns, two aggregated columns, one aggregation.
orders.groupby(["ship_mode", "customer_segment"], as_index=False)[['sales', 'discount']].mean()


In [ ]:
# .agg lets you ask for several aggregations at once: average and total sales per segment.
orders.groupby(["customer_segment"], as_index=False)['sales'].agg(['mean', 'sum'])


In [ ]:
# Named aggregates: new_column = ("source_column", "aggregation").
(orders
 .groupby("customer_segment", as_index=False)
 .agg(avg_sales=("sales", "mean"),
      max_discount=("discount", "max"),
      order_count=("row_id", "count"))
 )

**Why named aggregates win.** The output columns are called `avg_sales`, `max_discount` and `order_count` rather than a stack of `mean`/`max` headers you have to decode, and you can mix different aggregations on different columns in one call.

The wrapping parentheses are a formatting trick, not syntax: they let you put each step of the chain on its own line so the sequence reads top to bottom. You will see this style everywhere in production pandas code.


## Tips and Tricks with Groupby

1.  `as_index=False` flattens the group labels into ordinary columns, including when you group by several columns. What it does not flatten is the column headers: ask `.agg()` for several functions on several columns and you get two-level headers (`sales`/`mean`, `sales`/`sum`, …). Named aggregates avoid that.
1.  Getting aggregates for multiple columns is great, but you don't necessarily need to think in large tasks.  Break up your needs into smaller units of work, and we can combine them together (combining tables with joins and merges is session 8)
1.  `pd.crosstab` lets us interact multiple columns by allowing us to specify row/column definitions, along with an aggregation
1.  `.pivot_table` give us much more control over how we can aggregate our data.  While groupby is extremely powerful, there may be times when you want to have more control over your aggregation output.

In [ ]:
# crosstab: one variable down the rows, another across the columns, one aggregation in the cells.
pd.crosstab(index=orders.ship_mode, columns=orders.region,
            values=orders.sales, aggfunc="count")


> Same warning as before, in a new outfit: `aggfunc="count"` counts orders that have a sales figure, while leaving `values`/`aggfunc` off entirely counts every row in the cell. On this column they agree because `sales` is complete — on `product_base_margin` they would not.


In [ ]:
# pivot_table does the same job with more control -- several value columns at once, plus totals.
pd.pivot_table(orders, index="ship_mode", columns="region",
               values=["sales", "profit"], aggfunc="mean", margins=True).round(2)


> `margins=True` adds the "All" row and column. Note that the `All` figure is the mean over **all rows**, not the mean of the cells above it; averaging averages of unequal group sizes gives a different (and usually wrong) number.

**groupby, crosstab or pivot_table?** They compute the same numbers; they differ in shape. `groupby` returns one row per group — the long format that feeds plots and merges. `pivot_table` and `crosstab` return a grid — the wide format a human reads in a slide. `margins=True` adds the row and column totals that a director will ask for anyway.

Rule of thumb: **compute long with `groupby`, present wide with `pivot_table`.**


## Practice: the diamonds data

We are going to use the diamonds dataset loaded in the Setup section.

**Exercise 2.** For each combination of cut and color, get the average price and the number of rows, then sort so the most expensive combinations come first.

In [ ]:
# Your code here

**Answer**

*Provide your answer here*

# Mini Practice Problem: Data Request


Your firm has just begun to formally track and collect customer complaint data. The tracking system and process are new, so the data is not yet perfect.

You are on the data team and assigned a Jira ticket to help the Director of Customer Support. The director wants your help to understand the main sources of complaints and is looking for guidance on what changes should be prioritized first.

__Question:__ Based on your analysis of the complaint data, what is your recommendation for where the manager should spend their time?


In [ ]:
# Starter code: a list of dictionaries, which pd.DataFrame turns straight into a table.
data = [
    {"complaint_id": "C001", "category": "Card",    "description": "Charged twice",                "date_submitted": "2025-07-01", "tenure_months": 12,  "loyalty_program": "Silver",   "customer_spend_usd": 5000},
    {"complaint_id": "C002", "category": "Cards",   "description": "Card not received",            "date_submitted": "2025-07-02", "tenure_months": 24,  "loyalty_program": "Gold",     "customer_spend_usd": 12000},
    {"complaint_id": "C003", "category": "CRAD",    "description": "Card declined at POS",         "date_submitted": "2025-07-02", "tenure_months": None,"loyalty_program": None,       "customer_spend_usd": 2000},
    {"complaint_id": "C004", "category": "ATM",     "description": "Cash not dispensed",           "date_submitted": "2025-07-03", "tenure_months": 6,   "loyalty_program": "Bronze",   "customer_spend_usd": 1500},
    {"complaint_id": "C005", "category": "atm",     "description": "Machine out of service",       "date_submitted": "2025-07-03", "tenure_months": 8,   "loyalty_program": "Bronze",   "customer_spend_usd": 2000},
    {"complaint_id": "C006", "category": "Loans",   "description": "Late fee dispute",             "date_submitted": "2025-07-04", "tenure_months": None,"loyalty_program": None,       "customer_spend_usd": 800},
    {"complaint_id": "C007", "category": "Loan",    "description": "Payment not applied",          "date_submitted": "2025-07-05", "tenure_months": 18,  "loyalty_program": "Silver",   "customer_spend_usd": 7000},
    {"complaint_id": "C008", "category": "checking","description": "Unexpected monthly fee",       "date_submitted": "2025-07-06", "tenure_months": 3,   "loyalty_program": "Bronze",   "customer_spend_usd": 1000},
    {"complaint_id": "C009", "category": "Checking","description": "Overdraft fee issue",          "date_submitted": "2025-07-06", "tenure_months": None,"loyalty_program": None,       "customer_spend_usd": 500},
    {"complaint_id": "C010", "category": "Mortg.",  "description": "Escrow analysis confusion",    "date_submitted": "2025-07-07", "tenure_months": 30,  "loyalty_program": "Platinum", "customer_spend_usd": 25000},
    {"complaint_id": "C011", "category": "Mortgage","description": "Rate adjustment unclear",      "date_submitted": "2025-07-07", "tenure_months": 36,  "loyalty_program": "Platinum", "customer_spend_usd": 30000},
    {"complaint_id": "C012", "category": "Card",    "description": "Lost card replacement delay",  "date_submitted": "2025-07-08", "tenure_months": 5,   "loyalty_program": "Bronze",   "customer_spend_usd": 1200},
    {"complaint_id": "C013", "category": "ATM",     "description": "Deposit not posted",           "date_submitted": "2025-07-09", "tenure_months": None,"loyalty_program": None,       "customer_spend_usd": 700},
    {"complaint_id": "C014", "category": "Cards",   "description": "Incorrect card limit",         "date_submitted": "2025-07-10", "tenure_months": 20,  "loyalty_program": "Gold",     "customer_spend_usd": 11000},
    {"complaint_id": "C015", "category": "Loan",    "description": "Prepayment penalty confusion", "date_submitted": "2025-07-10", "tenure_months": 15,  "loyalty_program": "Silver",   "customer_spend_usd": 6000},
]

complaints = pd.DataFrame(data)
complaints


The cleaning here is last session's work, so we do it in one cell and move on. The question this time
is not "what is in the file" but "what does the director do on Monday" -- and that is an aggregation
question, because it depends on how the complaints stack up by category.

In [ ]:
# Step 1: last session's cleaning, compressed into one cell -- standardise the case, then map the
# typos onto one agreed spelling. Skip this and Card/Cards/CRAD count as three categories.
fixes = {'cards': 'card', 'crad': 'card', 'loans': 'loan', 'mortg.': 'mortgage'}

complaints['category_clean'] = complaints.category.str.lower().str.strip().replace(fixes)
complaints.category_clean.value_counts()

In [ ]:
# Your code here -- step 2: complaints and customer spend per category, in one table

In [ ]:
# Your code here -- step 3: rank the categories a second way, by money rather than by volume

In [ ]:
# Your code here -- step 4: how much of the customer context (tenure, loyalty) is actually usable?

**Recommendation**

*Provide your answer here*

# Follow up Questions and Exercises

**Question 1.** `groupby` gives you the group labels as the index by default, and `as_index=False` turns them back into ordinary columns. Why does that choice matter in practice?


**Answer**

*Provide your answer here*

**Question 2.** `.count()` and `.size()` returned different numbers for the shipping-mode groups. Which one would you use to compute "the share of orders that shipped by Regular Air", and why?


**Answer**

*Provide your answer here*

**Question 3.** We produced the same numbers with `groupby`, `crosstab` and `pivot_table`. How would you choose between them?


**Answer**

*Provide your answer here*

**Exercise 1.** Reshaping. Build a wide table of total sales by region (rows) and product category (columns) for the Sales Director, then turn it back into a long table you could feed to a chart.


In [ ]:
# Your code here -- step 1: the wide table (regions down the side, product categories across the top)

In [ ]:
# Your code here -- step 2: turn the wide table back into a long one

**Answer**

*Provide your answer here*

**Exercise 2.** Grain. Check the orders table for duplicate rows, then check whether `order_id` uniquely identifies a row. What you find decides what a `groupby` on this table is actually counting.

In [ ]:
# Your code here

**Answer**

*Provide your answer here*

**Exercise 3.** Put groupby to work on the survey data: does income rise with education? Rebuild last session's ordered categorical column first, and say how much you trust the answer.

In [ ]:
# The survey data, freshly loaded.
df2 = pd.read_csv(DATA + "sample-survey-data.csv")

# Your code here -- step 1: mark the impossible income missing, and rebuild Education as an ordered categorical

In [ ]:
# Your code here -- step 2: respondents and average income per education level, in education order

**Answer**

*Provide your answer here*